# 🎯 Notebook 3: Multi-Class Stimulus Classification (SVM & KNN)

## 1. Scientific Objective
Can machine learning models distinguish among multiple distinct environmental stimuli applied to plants?
We formulate a **4-class classification problem**:
1. **`normal` (Baseline):** Resting plant with ambient room noise and natural baseline drift.
2. **`music` (Acoustic Vibration):** Exposure to auditory pressure waves across musical tracks.
3. **`water` (Hydration Event):** Rapid ion conductance and cellular osmotic changes upon root watering.
4. **`dead` (Negative Control):** Dehydrated / non-viable tissue.

### Algorithms Tested:
- **Support Vector Machine (SVM):** Non-linear Radial Basis Function (RBF) kernel with Platt scaling probability estimates.
- **K-Nearest Neighbors (KNN):** Distance-weighted nearest-neighbor classifier in standardized temporal feature space.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv("plant_time_series_features.csv")
print("Target Stimulus Classes:")
print(df["main_category"].value_counts())


## 2. Feature Matrix & Stratified Session Grouping
We extract features and scale all features with `StandardScaler` (essential for distance-based SVM and KNN).


In [ ]:
feature_cols = [
    "mean", "std", "var", "min", "max", "range", "median", 
    "rms", "mad", "p25", "p75", "iqr", "zcr", "skew", "kurt", 
    "peak_count", "low_freq_power", "mid_freq_power", "high_freq_power", 
    "spectral_entropy", "dominant_freq"
]

class_labels = ["normal", "music", "water", "dead"]
label_to_idx = {l: i for i, l in enumerate(class_labels)}

X = df[feature_cols].values
y = df["main_category"].map(label_to_idx).values
groups = df["session_id"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Dataset prepared: {X_scaled.shape[0]} windows, {X_scaled.shape[1]} standardized features across 4 classes.")


## 3. Train Support Vector Machine (RBF Kernel)
SVM maximizes the margin of separation in the infinite-dimensional Hilbert space induced by the RBF kernel:
$$K(x, x') = \exp(-\gamma ||x - x'||^2)$$


In [ ]:
svm_model = SVC(kernel='rbf', C=5.0, gamma='scale', probability=True, random_state=42)
oof_preds_svm = np.zeros(len(y), dtype=int)
oof_proba_svm = np.zeros((len(y), len(class_labels)))

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X_scaled, y, groups)):
    X_tr, y_tr = X_scaled[train_idx], y[train_idx]
    X_te, y_te = X_scaled[test_idx], y[test_idx]
    
    svm_model.fit(X_tr, y_tr)
    oof_preds_svm[test_idx] = svm_model.predict(X_te)
    oof_proba_svm[test_idx] = svm_model.predict_proba(X_te)

print("=== Support Vector Machine (RBF) Performance ===")
print(classification_report(y, oof_preds_svm, target_names=class_labels))


## 4. Train K-Nearest Neighbors (KNN with Distance Weighting)
KNN classifies each window based on the majority vote of its $k$ closest temporal feature vectors, weighted inversely by Euclidean distance.


In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=7, weights='distance', n_jobs=-1)
oof_preds_knn = np.zeros(len(y), dtype=int)
oof_proba_knn = np.zeros((len(y), len(class_labels)))

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X_scaled, y, groups)):
    X_tr, y_tr = X_scaled[train_idx], y[train_idx]
    X_te, y_te = X_scaled[test_idx], y[test_idx]
    
    knn_model.fit(X_tr, y_tr)
    oof_preds_knn[test_idx] = knn_model.predict(X_te)
    oof_proba_knn[test_idx] = knn_model.predict_proba(X_te)

print("=== K-Nearest Neighbors (k=7) Performance ===")
print(classification_report(y, oof_preds_knn, target_names=class_labels))


## 5. Multi-Class Normalized Confusion Matrices
We examine misclassification patterns across stimuli.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

cm_svm = confusion_matrix(y, oof_preds_svm, normalize='true')
sns.heatmap(cm_svm, annot=True, fmt=".2%", cmap="Blues", ax=ax1,
            xticklabels=class_labels, yticklabels=class_labels)
ax1.set_title("Support Vector Machine (RBF Kernel)", fontweight='bold')
ax1.set_ylabel("True Stimulus")
ax1.set_xlabel("Predicted Stimulus")

cm_knn = confusion_matrix(y, oof_preds_knn, normalize='true')
sns.heatmap(cm_knn, annot=True, fmt=".2%", cmap="Greens", ax=ax2,
            xticklabels=class_labels, yticklabels=class_labels)
ax2.set_title("K-Nearest Neighbors (k=7, Distance-Weighted)", fontweight='bold')
ax2.set_ylabel("True Stimulus")
ax2.set_xlabel("Predicted Stimulus")

plt.tight_layout()
plt.show()


## 6. One-vs-Rest (OvR) ROC Curves for Each Stimulus Class
Evaluates sensitivity vs specificity individually for Normal, Music, Water, and Dead.


In [ ]:
plt.figure(figsize=(10, 6))
colors = ["#1976D2", "#7B1FA2", "#388E3C", "#D32F2F"]

for i, (c_name, color) in enumerate(zip(class_labels, colors)):
    y_binary = (y == i).astype(int)
    fpr, tpr, _ = roc_curve(y_binary, oof_proba_svm[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2.5, label=f'{c_name.capitalize()} vs Rest (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5)
plt.title("One-vs-Rest ROC Curves (SVM RBF)", fontweight='bold')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()


## 7. t-SNE 2D Manifold Visualization of Stimulus Space
t-Distributed Stochastic Neighbor Embedding (t-SNE) preserves local manifold topology, revealing how stimulus clusters organize in feature space.


In [ ]:
tsne = TSNE(n_components=2, perplexity=35, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
palette = {"normal": "#1976D2", "music": "#7B1FA2", "water": "#388E3C", "dead": "#D32F2F"}

for c_name in class_labels:
    idx = (df["main_category"] == c_name).values
    plt.scatter(X_tsne[idx, 0], X_tsne[idx, 1], 
                label=c_name.capitalize(), color=palette[c_name], 
                alpha=0.65, edgecolors='none', s=35)

plt.title("t-SNE 2D Manifold Projection of Plant Stimulus Clusters", fontweight='bold')
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.legend(framealpha=0.9, loc="upper right")
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()


## 8. KNN Hyperparameter Optimization (Tuning $k$)
We evaluate the validation accuracy across neighbor counts $k \in [1, 25]$.


In [ ]:
k_range = range(1, 25, 2)
k_scores = []

for k in k_range:
    knn_temp = KNeighborsClassifier(n_neighbors=k, weights='distance')
    # Fast 3-fold evaluation
    preds = cross_val_predict(knn_temp, X_scaled, y, cv=3)
    k_scores.append(np.mean(preds == y))

plt.figure(figsize=(9, 4.5))
plt.plot(k_range, [s*100 for s in k_scores], marker='o', color='#E65100', lw=2)
plt.title("KNN Accuracy vs. Number of Neighbors ($k$)", fontweight='bold')
plt.xlabel("Neighbor Count ($k$)")
plt.ylabel("Accuracy (%)")
plt.grid(True)
plt.tight_layout()
plt.show()
